<a href="https://colab.research.google.com/github/sreent/data-management-intro/blob/main/SPARQL%20for%20Graph%20Querying/09-lab-sparql-querying-knowledge-graphs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 9 — SPARQL: Querying the Movie Knowledge Graph

This lab is **code-based** — the first time you write executable queries against the knowledge graph you explored by hand in Chapter 8. For several queries you will recall your hand-traced traversal from Chapter 8, write the SPARQL equivalent, predict the bindings table, and verify by running the query. Queries use the `%%sparql` magic command from `cellspell`; graph-mutation exercises (adding triples for inference and enrichment) use `rdflib`.

Use the formal vocabulary (triple, binding, basic graph pattern, shared variable, OPTIONAL, FILTER, property path, open-world assumption) throughout your work.

**Verify-after-predict discipline.** After every query, compare the actual binding count to your prediction. If the count does not match, trace back through the Algorithm's Steps 2–3 to find where the mismatch occurred. Record the discrepancy and the diagnosis in your notebook — mismatches are learning opportunities, not failures.

**Materials required:** Google Colab notebook. The three Turtle files from Chapter 8 (`movie_ontology.ttl`, `source_a.ttl`, `source_b.ttl`) — downloaded automatically in the setup cell.

**Time budget:** Part 1 ~20 min | Part 2 ~30 min | Part 3 ~15 min | Part 4 ~20 min | Part 5 ~15 min | Part 6 ~10 min | Part 7 ~10 min | **Total ~120 min**

---

## Setup

Install dependencies, download Turtle files, merge them into a single graph file, and verify the setup.

In [1]:
# ── Cell 1: Install dependencies ───────────────────────
!pip install "cellspell[sparql] @ git+https://github.com/sreent/jupyter-query-magics.git" -q
!pip install -q rdflib
%load_ext cellspell.sparql

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 40.2 MB/s eta 0:00:00
✓ sparql spell loaded — %sparql, %%sparql


In [2]:
# ── Cell 2: Download Turtle files from course repository ─
import urllib.request, os

BASE_URL = "https://raw.githubusercontent.com/sreent/data-management-intro/main/resources/movie-kg/"
TTL_FILES = ["movie_ontology.ttl", "source_a.ttl", "source_b.ttl"]

for fname in TTL_FILES:
    if not os.path.exists(fname):
        urllib.request.urlretrieve(BASE_URL + fname, fname)
        print(f"Downloaded {fname} ({os.path.getsize(fname):,} bytes)")

# ── Merge into a single graph file for %%sparql magic ────
from rdflib import Graph
g_setup = Graph()
for fname in TTL_FILES:
    g_setup.parse(fname)
g_setup.serialize("movie_kg.ttl", format="turtle")
print(f"Merged {len(g_setup)} triples into movie_kg.ttl")
# Expected: approximately 128 triples (34 ontology + 58 source_a + 36 source_b)

Downloaded movie_ontology.ttl (2,060 bytes)
Downloaded source_a.ttl (4,173 bytes)
Downloaded source_b.ttl (2,708 bytes)
Merged 128 triples into movie_kg.ttl


In [3]:
# ── Cell 3: Verify setup ──────────────────────────────────
%%sparql --file movie_kg.ttl
PREFIX ex:   <http://example.org/movie#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT (COUNT(*) AS ?tripleCount) WHERE { ?s ?p ?o }
# Expected: one row with tripleCount around 128

✓ Loaded: movie_kg.ttl (+128 triples, 128 total)
tripleCount
-----------
128        

(1 row)



**Standard PREFIX block.** Include these declarations at the top of every `%%sparql` cell:

```sparql
PREFIX ex:   <http://example.org/movie#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>
```

---

## Entity URI Reference

Entity URIs in the movie KG are IMDB-style opaque identifiers — you need label triples (`ex:hasName`, `ex:hasTitle`) to know what they refer to. Keep this table open as you work.

| URI | Label | Type |
|---|---|---|
| `ex:person_nm0634240` | Christopher Nolan | Director |
| `ex:person_nm0898288` | Denis Villeneuve | Director |
| `ex:person_nm1950086` | Greta Gerwig | Director, Actor |
| `ex:person_nm3154303` | Timothée Chalamet | Actor |
| `ex:person_nm0614165` | Cillian Murphy | Actor |
| `ex:person_nm3053338` | Margot Robbie | Actor |
| `ex:film_tt0372784` | Batman Begins | Film |
| `ex:film_tt0468569` | The Dark Knight | Film |
| `ex:film_tt1345836` | The Dark Knight Rises | Film |
| `ex:film_tt1375666` | Inception | Film |
| `ex:film_tt0816692` | Interstellar | Film |
| `ex:film_tt15398776` | Oppenheimer | Film |
| `ex:film_tt1160419` | Dune | Film |
| `ex:film_tt4925292` | Lady Bird | Film |
| `ex:film_tt1517268` | Barbie | Film |
| `ex:genre_scifi` | Science Fiction | Genre |
| `ex:genre_action` | Action | Genre |
| `ex:genre_drama` | Drama | Genre |
| `ex:genre_crime` | Crime | Genre |
| `ex:genre_comedy` | Comedy | Genre |
| `ex:genre_biography` | Biography | Genre |
| `ex:award_bestpicture` | Academy Award for Best Picture | Award |
| `ex:award_bestdirector` | Academy Award for Best Director | Award |
| `ex:award_bestactor` | Academy Award for Best Actor | Award |

Note the contrast: predicates like `ex:directed`, `ex:hasGenre`, `ex:actedIn` are readable because they are vocabulary — part of the ontology. Entity URIs are opaque because they are data — instances, not definitions.

---

## Part 1 — Manual Trace (Building the Mental Model on Paper)

This is the centrepiece of the lab. Before writing any code, you complete bindings tables by hand — the single most effective exercise for building the SPARQL mental model. Do this on paper or in a text cell, **not** by running a query.

### Q1. Manual bindings trace — printed subgraph

The following subgraph contains 12 triples:

```turtle
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix ex:  <http://example.org/movie#> .

ex:person_nm0634240  rdf:type     ex:Director ;
                     ex:hasName   "Christopher Nolan" ;
                     ex:directed  ex:film_tt1375666 ;
                     ex:directed  ex:film_tt0816692 .

ex:film_tt1375666    rdf:type     ex:Film ;
                     ex:hasTitle  "Inception" ;
                     ex:hasGenre  ex:genre_scifi ;
                     ex:hasGenre  ex:genre_action .

ex:film_tt0816692    rdf:type     ex:Film ;
                     ex:hasTitle  "Interstellar" ;
                     ex:hasGenre  ex:genre_scifi ;
                     ex:hasGenre  ex:genre_drama .
```

Given this SPARQL query:

```sparql
SELECT ?filmTitle ?genre
WHERE {
  ex:person_nm0634240 ex:directed ?film .
  ?film ex:hasTitle ?filmTitle .
  ?film ex:hasGenre ?genre .
}
```

Complete the bindings table by hand, working pattern by pattern:

**(a)** After pattern 1 (`ex:person_nm0634240 ex:directed ?film`), list the bindings for `?film`:

| ?film |
|---|
| ? |
| ? |

**(b)** After pattern 2 (`?film ex:hasTitle ?filmTitle`), extend the table. Does the row count change? Why or why not?

| ?film | ?filmTitle |
|---|---|
| ? | ? |
| ? | ? |

**(c)** After pattern 3 (`?film ex:hasGenre ?genre`), extend the table. Does the row count change? How many rows now? Why?

| ?film | ?filmTitle | ?genre |
|---|---|---|
| ? | ? | ? |
| ... | ... | ... |

**(d)** The query SELECTs `?filmTitle` and `?genre`, but `?genre` is a URI — not a human-readable name. What would the output actually show for the genre column? What pattern would you need to add to get readable genre names?

**(e)** If we change to `SELECT DISTINCT ?filmTitle`, how many rows would the result have? If we change to `SELECT (COUNT(DISTINCT ?film) AS ?filmCount)` (no GROUP BY), what would the count be?

<details>
<summary>Solution</summary>

**(a)** Two bindings — one for each `ex:directed` edge from Nolan in this subgraph:

| ?film |
|---|
| ex:film_tt1375666 |
| ex:film_tt0816692 |

**(b)** Still 2 rows. `ex:hasTitle` is a 1:1 property — each film has exactly one title. No multiplication:

| ?film | ?filmTitle |
|---|---|
| ex:film_tt1375666 | "Inception" |
| ex:film_tt0816692 | "Interstellar" |

**(c)** Now 4 rows. `ex:hasGenre` is 1:many — Inception has 2 genres and Interstellar has 2 genres. Each film's bindings are multiplied by its genre count:

| ?film | ?filmTitle | ?genre |
|---|---|---|
| ex:film_tt1375666 | "Inception" | ex:genre_scifi |
| ex:film_tt1375666 | "Inception" | ex:genre_action |
| ex:film_tt0816692 | "Interstellar" | ex:genre_scifi |
| ex:film_tt0816692 | "Interstellar" | ex:genre_drama |

**(d)** The genre column shows URIs: `http://example.org/movie#genre_scifi`, `http://example.org/movie#genre_action`, etc. To get readable names, add the pattern `?genre ex:hasName ?genreName` and select `?genreName` instead.

**(e)** `SELECT DISTINCT ?filmTitle` returns 2 rows ("Inception", "Interstellar") — the 4 rows collapse to 2 unique titles. `COUNT(DISTINCT ?film)` returns 1 row with value 2 — two distinct film URIs appear in the bindings.

</details>

---

### Q2. Manual trace — OPTIONAL

Using the same 12-triple subgraph from Q1, add this triple:

```turtle
ex:film_tt1375666  ex:wonAward  ex:award_bestpicture .
```

Given this query:

```sparql
SELECT ?filmTitle ?award
WHERE {
  ex:person_nm0634240 ex:directed ?film .
  ?film ex:hasTitle ?filmTitle .
  OPTIONAL { ?film ex:wonAward ?award }
}
```

**(a)** Complete the bindings table by hand. Which rows have `?award` bound? Which have `?award` unbound?

**(b)** Now add `FILTER(BOUND(?award))` after the OPTIONAL. Which rows survive? How many rows now? What happened to Interstellar?

**(c)** Now change to `FILTER(!BOUND(?award))`. Which rows survive? What question does this answer?

<details>
<summary>Solution</summary>

**(a)** Two rows from the mandatory patterns (directed → hasTitle). The OPTIONAL adds `?award` where a match exists:

| ?film | ?filmTitle | ?award |
|---|---|---|
| ex:film_tt1375666 | "Inception" | ex:award_bestpicture |
| ex:film_tt0816692 | "Interstellar" | (unbound) |

Inception has a `wonAward` edge → `?award` is bound. Interstellar has no `wonAward` edge → `?award` is unbound (not dropped — OPTIONAL preserves the row).

**(b)** `FILTER(BOUND(?award))` keeps only rows where `?award` is bound → 1 row (Inception). Interstellar is dropped because its `?award` is unbound. This converts the LEFT JOIN into an INNER JOIN.

**(c)** `FILTER(!BOUND(?award))` keeps only rows where `?award` is unbound → 1 row (Interstellar). This answers: "Which films have no award data in this subgraph?" — the anti-join pattern.

</details>

---

## Part 2 — From Hand-Traced Traversals to SPARQL

For each query below, recall the hand-traced traversal from Chapter 8, write the SPARQL equivalent, predict the bindings table, and verify by running the query.

### Q3. Single hop — Chapter 8 revisited

In Chapter 8, you started at `ex:person_nm0634240` on your printed graph drawing and followed all outgoing `ex:directed` edges by hand, resolving each film's title to fill in a table. Now express that same traversal as a SPARQL query.

**Before running, predict:** How many rows should the result have? (Same as the number of `ex:directed` edges from Nolan — count them in `source_a.ttl`.)

In [4]:
%%sparql --file movie_kg.ttl
PREFIX ex:   <http://example.org/movie#>

SELECT ?title WHERE {
  ex:person_nm0634240 ex:directed ?film .
  ?film ex:hasTitle ?title .
}

title                
---------------------
Batman Begins        
The Dark Knight      
Interstellar         
The Dark Knight Rises
Inception            
Oppenheimer          

(6 rows)



Run and verify. Compare: Chapter 8's hand-traced version required you to start at Nolan on your graph drawing and follow every `ex:directed` edge, then hop to each film to read its label. The SPARQL version is 2 patterns in a WHERE clause — the engine does the scanning. Both produce the same result.

<details>
<summary>Solution</summary>

**Predicted:** 6 rows — Nolan has 6 `ex:directed` edges in `source_a.ttl`.

**Actual:** 6 rows:

| ?title |
|---|
| "Inception" |
| "Interstellar" |
| "Oppenheimer" |
| "Batman Begins" |
| "The Dark Knight" |
| "The Dark Knight Rises" |

The `ex:hasTitle` pattern is 1:1, so the row count equals the number of `directed` edges.

</details>

---

### Q4. Type filter and multi-role — Chapter 8 revisited

In Chapter 8, you drew the class hierarchy and identified the entity with two `rdf:type` declarations (Greta Gerwig — both Director and Actor). Now express equivalent queries in SPARQL.

**(a)** All directors with names:

In [5]:
%%sparql --file movie_kg.ttl
PREFIX ex:   <http://example.org/movie#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?name WHERE {
  ?person rdf:type ex:Director .
  ?person ex:hasName ?name .
}

name             
-----------------
Christopher Nolan
Denis Villeneuve 
Greta Gerwig     

(3 rows)



**Predict:** How many results?

**(b)** All actors with names. Write the query yourself and predict the count.

**(c)** People who are *both* director and actor (the intersection):

In [6]:
%%sparql --file movie_kg.ttl
PREFIX ex:   <http://example.org/movie#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?name WHERE {
  ?person rdf:type ex:Director .
  ?person rdf:type ex:Actor .
  ?person ex:hasName ?name .
}

name        
------------
Greta Gerwig

(1 row)



**Predict:** How many results for (c)? Compare to the dual-type entity you identified in Chapter 8. The shared `?person` variable acts as an intersection — only entities that match *both* type patterns survive.

<details>
<summary>Solution</summary>

**(a)** 3 results — Nolan, Villeneuve, Gerwig (all typed `ex:Director`).

**(b)** The query:

```sparql
SELECT ?name WHERE {
  ?person rdf:type ex:Actor .
  ?person ex:hasName ?name .
}
```

4 results — Chalamet, Murphy, Robbie, and Gerwig (who is typed as both Director and Actor).

**(c)** 1 result — "Greta Gerwig". She is the only entity with both `rdf:type ex:Director` and `rdf:type ex:Actor`. The shared `?person` variable ensures both type patterns must match the same URI.

</details>

---

### Q5. Two-hop integration — cross-source

Chapter 8's cross-source integration test: starting at Nolan, follow `ex:directed` edges to films (from `source_a.ttl`), then follow `ex:hasGenre` edges to genres (from `source_b.ttl`), resolving each genre's `ex:hasName`.

Write the SPARQL equivalent. **Before running, predict:**

- How many bindings before adding the genre pattern? (= number of Nolan films with `ex:directed` edges)
- How many bindings after? (= sum of `ex:hasGenre` edges across those films — note: not all Nolan films have genre data in `source_b.ttl`; films without genres are *dropped* by this inner pattern.)
- How many `DISTINCT ?genreName` values?

In [7]:
%%sparql --file movie_kg.ttl
PREFIX ex:   <http://example.org/movie#>

SELECT ?genreName WHERE {
  ex:person_nm0634240 ex:directed ?film .
  ?film ex:hasGenre ?genre .
  ?genre ex:hasName ?genreName .
}

genreName      
---------------
Action         
Crime          
Drama          
Drama          
Science Fiction
Action         
Science Fiction
Biography      
Drama          

(9 rows)



Run and count rows. Compare to prediction. Then add `DISTINCT` and compare. In Chapter 8's hand-traced version, you naturally collected unique genre names. SPARQL returns a bag — you must choose `DISTINCT` explicitly. Which is the right answer to "what genres does Nolan work in?" — the deduplicated list or the full bindings table?

<details>
<summary>Solution</summary>

**Before genre pattern:** 6 bindings (6 films Nolan directed).

**After genre pattern:** Only 4 of Nolan's 6 films have `ex:hasGenre` data in `source_b.ttl`. Batman Begins and The Dark Knight Rises have no genre triples — they are dropped by the inner join.

Genre counts for the 4 remaining films:
- Inception: 2 (Sci-Fi, Action)
- Interstellar: 2 (Sci-Fi, Drama)
- Oppenheimer: 2 (Drama, Biography)
- The Dark Knight: 3 (Action, Crime, Drama)

Total: 2 + 2 + 2 + 3 = **9 rows**.

**DISTINCT ?genreName:** 5 values — Science Fiction, Action, Drama, Biography, Crime.

The full bindings table (9 rows) tells you how many films connect to each genre. The deduplicated list (5 rows) answers "what genres?" Both are valid — the right one depends on the question.

</details>

---

### Q6. Multi-hop intersection — Chapter 8 revisited

Chapter 8's centrepiece: "actors who appeared in films directed by both Nolan and Villeneuve." You traced it in five explicit steps on paper — collecting Nolan's films, collecting Villeneuve's films, finding actors in each set, then intersecting. The SPARQL version expresses all five steps as a single pattern:

In [8]:
%%sparql --file movie_kg.ttl
PREFIX ex:   <http://example.org/movie#>

SELECT DISTINCT ?actorName WHERE {
  ex:person_nm0634240 ex:directed ?nolanFilm .
  ex:person_nm0898288  ex:directed ?villeneuveFilm .
  ?actor ex:actedIn ?nolanFilm .
  ?actor ex:actedIn ?villeneuveFilm .
  ?actor ex:hasName ?actorName .
}

actorName        
-----------------
Timothée Chalamet

(1 row)



**Before running, trace the bindings mentally:**

- Pattern 1: `?nolanFilm` — how many bindings?
- Pattern 2: `?villeneuveFilm` — how many? Since no shared variable with pattern 1, this is a cross-product.
- Patterns 3–4: `?actor` joins both — only actors appearing in *both* a Nolan and a Villeneuve film survive.
- Pattern 5: `?actorName` — 1:1.

**Predict:** How many rows without `DISTINCT`? Run with and without DISTINCT. Explain any difference.

<details>
<summary>Solution</summary>

- Pattern 1: `?nolanFilm` — 6 bindings (Nolan directed 6 films).
- Pattern 2: `?villeneuveFilm` — 1 binding (Villeneuve directed Dune). Cross-product: 6 × 1 = 6 combinations.
- Patterns 3–4: `?actor` must appear in BOTH a Nolan film AND a Villeneuve film. From the data: Chalamet acted in Interstellar (Nolan) and Dune (Villeneuve). Murphy acted in 3 Nolan films but no Villeneuve films. Robbie acted in Barbie (Gerwig), not Nolan or Villeneuve. Only Chalamet survives.
- Pattern 5: 1:1 — Chalamet has one `ex:hasName`.

**Result:** 1 row — "Timothée Chalamet".

Without DISTINCT: still 1 row in this dataset (Chalamet is in 1 Nolan film × 1 Villeneuve film = 1 combination). DISTINCT makes no difference here, but would matter if an actor appeared in multiple films from both directors.

</details>

---

### Q7. URIs everywhere — why graphs need labels

Run Q5's genre query but *remove* the name patterns — select the raw variables instead:

In [9]:
%%sparql --file movie_kg.ttl
PREFIX ex:   <http://example.org/movie#>

SELECT ?film ?genre WHERE {
  ex:person_nm0634240 ex:directed ?film .
  ?film ex:hasGenre ?genre .
}

film                                     | genre                                   
-----------------------------------------+-----------------------------------------
http://example.org/movie#film_tt0468569  | http://example.org/movie#genre_action   
http://example.org/movie#film_tt0468569  | http://example.org/movie#genre_crime    
http://example.org/movie#film_tt0468569  | http://example.org/movie#genre_drama    
http://example.org/movie#film_tt0816692  | http://example.org/movie#genre_drama    
http://example.org/movie#film_tt0816692  | http://example.org/movie#genre_scifi    
http://example.org/movie#film_tt1375666  | http://example.org/movie#genre_action   
http://example.org/movie#film_tt1375666  | http://example.org/movie#genre_scifi    
http://example.org/movie#film_tt15398776 | http://example.org/movie#genre_biography
http://example.org/movie#film_tt15398776 | http://example.org/movie#genre_drama    

(9 rows)



Look at the output. Every value is a URI: `http://example.org/movie#film_tt1375666`, `http://example.org/movie#genre_scifi`. You cannot read the film's title or the genre's name from the URI alone.

**(a)** Chapter 8's ontology flagged this friction explicitly: `ex:hasName` has no domain constraint — it can be used on any entity. `ex:hasTitle` is specific to films. To look up a human-readable display name, you need to know which property to use. RDF has a built-in standard for giving URIs human-readable names: `rdfs:label`, defined in the RDFS vocabulary. Wikidata, DBpedia, and virtually every public linked-data source use `rdfs:label`. Our graph uses domain-specific predicates instead. Discuss: why does this matter for interoperability?

**(b)** Add two ontology-level declarations to the graph and save the updated file:

In [10]:
from rdflib import Graph, Namespace, RDFS

g = Graph()
g.parse("movie_kg.ttl")
ex = Namespace("http://example.org/movie#")

g.add((ex.hasName, RDFS.subPropertyOf, RDFS.label))
g.add((ex.hasTitle, RDFS.subPropertyOf, RDFS.label))

g.serialize("movie_kg.ttl", format="turtle")
print("Added rdfs:subPropertyOf declarations to movie_kg.ttl")

Added rdfs:subPropertyOf declarations to movie_kg.ttl



This says: "`ex:hasName` is a sub-property of `rdfs:label`" — anything with an `ex:hasName` also has an `rdfs:label`. Same for `ex:hasTitle`. This is the same pattern as class hierarchies (`rdfs:subClassOf`) from Chapter 8 — but applied to properties instead of classes.

**(c)** Test whether the declaration works. Query for `rdfs:label` *without* inference:

In [11]:
%%sparql --file movie_kg.ttl
PREFIX ex:   <http://example.org/movie#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?label WHERE {
  ex:person_nm0634240 rdfs:label ?label .
}

(no results)



**Predict:** How many results? (Your graph has `ex:hasName "Christopher Nolan"` but no `rdfs:label` triple. The `rdfs:subPropertyOf` declaration exists, but the SPARQL engine does not apply RDFS inference by default.)

Now enable inference by manually expanding `rdfs:subPropertyOf` and saving the updated graph:

In [12]:
from rdflib import Graph, Namespace, RDFS

g = Graph()
g.parse("movie_kg.ttl")

# For each subPropertyOf declaration, materialise inferred triples
for sub, _, sup in g.triples((None, RDFS.subPropertyOf, None)):
    for s, _, o in list(g.triples((None, sub, None))):
        g.add((s, sup, o))

g.serialize("movie_kg.ttl", format="turtle")
inferred_count = len(list(g.triples((None, RDFS.label, None))))
print(f"After inference: {inferred_count} rdfs:label triples in movie_kg.ttl")

After inference: 24 rdfs:label triples in movie_kg.ttl



Re-run the `rdfs:label` query. **Predict:** How many results for `ex:person_nm0634240`?

**(d)** *[Extension — if time permits.]* Write a query that retrieves readable labels for *all* entities using only `rdfs:label` — no domain-specific predicates:

In [13]:
%%sparql --file movie_kg.ttl
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?entity ?label WHERE {
  ?entity rdfs:label ?label .
}
ORDER BY ?entity

(no results)



After inference, this returns names from `ex:hasName` *and* titles from `ex:hasTitle` — a single standard predicate retrieves labels from all domain predicates. This is the interoperability payoff.

**(e)** *[Extension.]* Discussion: Compare two approaches to resolving the `ex:hasName` vs `rdfs:label` gap:

| Approach | What it does | Trade-off |
|---|---|---|
| **Replace** `ex:hasName` with `rdfs:label` | Remove domain predicates, use only the standard | Loses domain specificity (cannot distinguish person names from film titles) |
| **Declare** `ex:hasName rdfs:subPropertyOf rdfs:label` | Keep domain predicates, link them to the standard | Requires RDFS inference; queries using `rdfs:label` only work if inference is enabled |

Which approach does real-world linked data use? (The declare approach — Dublin Core declares `dc:title rdfs:subPropertyOf rdfs:label`; FOAF declares `foaf:name rdfs:subPropertyOf rdfs:label`.)

<details>
<summary>Solution</summary>

**(a)** It matters because external consumers (search engines, data integration tools) look for `rdfs:label` by convention. If your graph uses only `ex:hasName` and `ex:hasTitle`, external tools cannot read entity names without being taught your custom predicates. Standard vocabularies enable interoperability without prior agreement on domain-specific properties.

**(b)** See code cell above.

**(c)** Before inference: **0 results**. The `rdfs:subPropertyOf` declaration exists, but SPARQL does not automatically infer that `ex:hasName` values also satisfy `rdfs:label` queries. After inference: **1 result** — `"Christopher Nolan"`. The materialisation step added an explicit `rdfs:label` triple for every entity that had `ex:hasName` or `ex:hasTitle`.

**(d)** After inference, this query returns one row per entity that has a name or title — all persons (via `ex:hasName → rdfs:label`), all films (via `ex:hasTitle → rdfs:label`), all genres and awards (via `ex:hasName → rdfs:label`). Total: 24 rows (matching the entity URI reference table).

**(e)** Real-world linked data uses the **declare** approach. FOAF declares `foaf:name rdfs:subPropertyOf rdfs:label`; Dublin Core declares `dc:title rdfs:subPropertyOf rdfs:label`. This preserves domain specificity while enabling interoperability — but only for consumers that materialise or reason over `rdfs:subPropertyOf`.

</details>

---

## Part 3 — Multiplicity and Aggregation

### Q8. COUNT inflation — multiplicity diagnosis

Write a query to count how many films each director directed:

In [14]:
%%sparql --file movie_kg.ttl
PREFIX ex:   <http://example.org/movie#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?directorName (COUNT(?film) AS ?filmCount)
WHERE {
  ?director rdf:type ex:Director .
  ?director ex:hasName ?directorName .
  ?director ex:directed ?film .
}
GROUP BY ?director ?directorName
ORDER BY DESC(?filmCount)

directorName      | filmCount
------------------+----------
Christopher Nolan | 6        
Greta Gerwig      | 2        
Denis Villeneuve  | 1        

(3 rows)



**Predict:** Does this count correctly? (Each `directed` edge is counted once per film. No pattern introduces 1:many multiplication *after* the `?film` binding.)

Now modify the query to also include genres — add `?film ex:hasGenre ?genre .` inside the WHERE block.

**Before running:** Predict what happens to the count. (Each film with multiple genres now contributes multiple bindings instead of one. The count is inflated by genre multiplication.)

Run both versions. Fix with `COUNT(DISTINCT ?film)`. Record both counts side by side.

This is Chapter 4's COUNT inflation (from joining through OrderLine), replayed in SPARQL. The mechanism is identical: a 1:many pattern multiplied bindings before COUNT saw them.

<details>
<summary>Solution</summary>

**Without genre pattern (correct):**

| ?directorName | ?filmCount |
|---|---|
| "Christopher Nolan" | 6 |
| "Greta Gerwig" | 2 |
| "Denis Villeneuve" | 1 |

**With genre pattern added (inflated):**

| ?directorName | ?filmCount |
|---|---|
| "Christopher Nolan" | 9 |
| "Denis Villeneuve" | 3 |
| "Greta Gerwig" | 3 |

Nolan: only 4 of 6 films have genre data (Batman Begins and Dark Knight Rises have none — dropped by inner join). Those 4 films produce 2+2+2+3 = 9 bindings. Villeneuve: Dune has 3 genres → 3. Gerwig: Lady Bird (2 genres) + Barbie (1 genre) = 3.

**Fixed with COUNT(DISTINCT ?film):**

| ?directorName | ?filmCount |
|---|---|
| "Christopher Nolan" | 4 |
| "Greta Gerwig" | 2 |
| "Denis Villeneuve" | 1 |

Note: Nolan drops from 6 to 4 because the genre inner join excluded 2 films. To get all 6, use OPTIONAL for the genre pattern.

</details>

---

### Q9. Aggregation — genre statistics

"For each genre, how many films belong to it, and how many distinct directors have made a film in that genre?"

In [15]:
%%sparql --file movie_kg.ttl
PREFIX ex:   <http://example.org/movie#>

SELECT ?genreName
       (COUNT(DISTINCT ?film) AS ?filmCount)
       (COUNT(DISTINCT ?director) AS ?directorCount)
WHERE {
  ?film ex:hasGenre ?genre .
  ?genre ex:hasName ?genreName .
  ?director ex:directed ?film .
}
GROUP BY ?genre ?genreName
ORDER BY DESC(?filmCount)

genreName       | filmCount | directorCount
----------------+-----------+--------------
Drama           | 5         | 3            
Science Fiction | 3         | 2            
Action          | 3         | 2            
Comedy          | 2         | 1            
Biography       | 1         | 1            
Crime           | 1         | 1            

(6 rows)



**Predict:** The query touches three entities (`?film`, `?genre`, `?director`). The shared variable `?film` links genres to directors through the `directed` relationship. Without `DISTINCT`, a film with 2 directors (if any exist) would inflate the film count per genre. With `DISTINCT`, counts are correct.

Discuss: why must *both* COUNTs use DISTINCT? What happens if you remove DISTINCT from only one?

<details>
<summary>Solution</summary>

| ?genreName | ?filmCount | ?directorCount |
|---|---|---|
| "Drama" | 5 | 3 |
| "Science Fiction" | 3 | 2 |
| "Action" | 3 | 2 |
| "Comedy" | 2 | 1 |
| "Crime" | 1 | 1 |
| "Biography" | 1 | 1 |

Drama is the most common genre (Interstellar, Oppenheimer, Dark Knight, Dune, Lady Bird = 5 films by 3 directors: Nolan, Villeneuve, Gerwig).

Both COUNTs need DISTINCT because the join between films and directors can multiply. For example, if a hypothetical film had 2 directors and 3 genres, removing DISTINCT from COUNT(?film) would count that film once per (director, genre) combination instead of once per genre.

</details>

---

## Part 4 — OPTIONAL, FILTER, and the Open World

### Q10. OPTIONAL — enrichment without filtering

List all films with their titles, and optionally their award names:

In [16]:
%%sparql --file movie_kg.ttl
PREFIX ex:   <http://example.org/movie#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?filmTitle ?awardName WHERE {
  ?film rdf:type ex:Film .
  ?film ex:hasTitle ?filmTitle .
  OPTIONAL {
    ?film ex:wonAward ?award .
    ?award ex:hasName ?awardName .
  }
}
ORDER BY ?filmTitle

filmTitle             | awardName                      
----------------------+--------------------------------
Barbie                |                                
Batman Begins         |                                
Dune                  |                                
Inception             |                                
Interstellar          |                                
Lady Bird             |                                
Oppenheimer           | Academy Award for Best Actor   
Oppenheimer           | Academy Award for Best Director
Oppenheimer           | Academy Award for Best Picture 
The Dark Knight       |                                
The Dark Knight Rises |                                

(11 rows)



**Before running:** Predict which films will have `?awardName` bound and which will have it unbound. Count: how many total rows? (Films with awards may have multiple rows — one per award. Films without awards have exactly one row with unbound `?awardName`.)

Run and verify predictions. Identify the unbound rows.

<details>
<summary>Solution</summary>

9 films total. Only Oppenheimer has `wonAward` triples (3 awards: Best Picture, Best Director, Best Actor). Oppenheimer produces 3 rows (one per award). The other 8 films produce 1 row each with `?awardName` unbound.

**Total: 11 rows** (8 unbound + 3 bound).

| ?filmTitle | ?awardName |
|---|---|
| "Barbie" | (unbound) |
| "Batman Begins" | (unbound) |
| "Dune" | (unbound) |
| "Inception" | (unbound) |
| "Interstellar" | (unbound) |
| "Lady Bird" | (unbound) |
| "Oppenheimer" | "Academy Award for Best Actor" |
| "Oppenheimer" | "Academy Award for Best Director" |
| "Oppenheimer" | "Academy Award for Best Picture" |
| "The Dark Knight" | (unbound) |
| "The Dark Knight Rises" | (unbound) |

</details>

---

### Q11. The FILTER trap — OPTIONAL interaction

Starting from Q10's query, you want to exclude Best Picture. Add a FILTER *outside* the OPTIONAL:

In [17]:
%%sparql --file movie_kg.ttl
PREFIX ex:   <http://example.org/movie#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?filmTitle ?awardName WHERE {
  ?film rdf:type ex:Film .
  ?film ex:hasTitle ?filmTitle .
  OPTIONAL {
    ?film ex:wonAward ?award .
    ?award ex:hasName ?awardName .
  }
  FILTER(?awardName != "Academy Award for Best Picture")
}
ORDER BY ?filmTitle

filmTitle   | awardName                      
------------+--------------------------------
Oppenheimer | Academy Award for Best Actor   
Oppenheimer | Academy Award for Best Director

(2 rows)



**Before running:** Predict which rows survive. Count the predicted result rows.

Run the query. Then answer:

- Did films without awards survive? (No — unbound `?awardName` caused the FILTER to error → drop the row.)
- How many films were lost compared to Q10?
- Write the fixed version that keeps films without awards: either move the FILTER inside the OPTIONAL, or use `FILTER(!BOUND(?awardName) || ?awardName != "Academy Award for Best Picture")`.

Run the fixed version and confirm all films survive.

<details>
<summary>Solution</summary>

**Broken query result:** 2 rows — only Oppenheimer's Best Director and Best Actor survive. All 8 films without awards are dropped because `unbound != "Academy Award for Best Picture"` evaluates to an error, which SPARQL treats as false. This is identical to Chapter 4's WHERE after LEFT JOIN dropping NULL-padded rows.

**Films lost:** 8 (all films without awards) + 1 (Oppenheimer's Best Picture row) = 9 rows lost from Q10's 11.

**Fix — move FILTER inside OPTIONAL:**

```sparql
SELECT ?filmTitle ?awardName WHERE {
  ?film rdf:type ex:Film .
  ?film ex:hasTitle ?filmTitle .
  OPTIONAL {
    ?film ex:wonAward ?award .
    ?award ex:hasName ?awardName .
    FILTER(?awardName != "Academy Award for Best Picture")
  }
}
ORDER BY ?filmTitle
```

**Or — guard the FILTER with BOUND:**

```sparql
FILTER(!BOUND(?awardName) || ?awardName != "Academy Award for Best Picture")
```

Both fixes produce **10 rows**: 8 unbound + Oppenheimer's Best Director and Best Actor.

</details>

---

### Q12. FILTER NOT EXISTS — the anti-join

"Which films in the graph have no award data?"

In [18]:
%%sparql --file movie_kg.ttl
PREFIX ex:   <http://example.org/movie#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?filmTitle WHERE {
  ?film rdf:type ex:Film .
  ?film ex:hasTitle ?filmTitle .
  FILTER NOT EXISTS { ?film ex:wonAward ?award }
}

filmTitle            
---------------------
The Dark Knight Rises
Lady Bird            
Batman Begins        
Interstellar         
Dune                 
Inception            
The Dark Knight      
Barbie               

(8 rows)



**Predict:** How many results? Run and verify.

**Discussion:** In SQL (closed world), this definitively means "films that won no awards." In SPARQL (open world), it means "films for which *this graph* has no award triples." Inception actually won 4 Academy Awards — if your graph does not include that data, it appears in this list. Write both interpretations and note which is correct for RDF.

<details>
<summary>Solution</summary>

**8 results** — all films except Oppenheimer:

| ?filmTitle |
|---|
| "Barbie" |
| "Batman Begins" |
| "Dune" |
| "Inception" |
| "Interstellar" |
| "Lady Bird" |
| "The Dark Knight" |
| "The Dark Knight Rises" |

**Closed-world interpretation (SQL):** "These 8 films won no awards." — Incorrect for RDF.

**Open-world interpretation (SPARQL):** "This graph contains no award triples for these 8 films." — Correct. Inception won 4 Oscars in reality, but our graph lacks that data. The absence means "unknown," not "untrue." This is the practical consequence of Chapter 8's open-world assumption.

</details>

---

## Part 5 — Property Paths (Multi-Hop Traversal)

### Q13. Sequel chain — Chapter 8 revisited

Chapter 8's sequel chain traced the Dark Knight trilogy by following `ex:sequelOf` edges on the graph drawing. Write the SPARQL equivalents:

**(a)** Direct prequel (one hop):

In [19]:
%%sparql --file movie_kg.ttl
PREFIX ex:   <http://example.org/movie#>

SELECT ?title WHERE {
  ex:film_tt1345836 ex:sequelOf ?prequel .
  ?prequel ex:hasTitle ?title .
}

title          
---------------
The Dark Knight

(1 row)



**Predict:** 1 result (The Dark Knight). Run and verify.

**(b)** All prequels (transitive, one or more hops):

In [20]:
%%sparql --file movie_kg.ttl
PREFIX ex:   <http://example.org/movie#>

SELECT ?title WHERE {
  ex:film_tt1345836 ex:sequelOf+ ?prequel .
  ?prequel ex:hasTitle ?title .
}

title          
---------------
The Dark Knight
Batman Begins  

(2 rows)



**Predict:** 2 results (The Dark Knight + Batman Begins). Run and verify.

**(c)** Bounded to exactly 2 hops:

In [21]:
%%sparql --file movie_kg.ttl
PREFIX ex:   <http://example.org/movie#>

SELECT ?title WHERE {
  ex:film_tt1345836 ex:sequelOf{2} ?prequel .
  ?prequel ex:hasTitle ?title .
}

SPARQL error: Expected SelectQuery, found 'ex'  (at char 66), (line:4, col:3)



**Predict:** 1 result (Batman Begins — two hops from Rises). Run and verify.

**(d)** All films in the trilogy, including the starting film (zero or more hops):

In [22]:
%%sparql --file movie_kg.ttl
PREFIX ex:   <http://example.org/movie#>

SELECT ?title WHERE {
  ex:film_tt1345836 ex:sequelOf* ?film .
  ?film ex:hasTitle ?title .
}

title                
---------------------
The Dark Knight Rises
The Dark Knight      
Batman Begins        

(3 rows)



**Predict:** 3 results (Rises itself + Dark Knight + Begins). Run and verify. Explain the difference between `+` (one or more) and `*` (zero or more).

<details>
<summary>Solution</summary>

**(a)** 1 result: "The Dark Knight" — the direct `sequelOf` edge from Dark Knight Rises.

**(b)** 2 results: "The Dark Knight" (1 hop) and "Batman Begins" (2 hops via Dark Knight). The `+` operator follows one or more hops transitively.

**(c)** 1 result: "Batman Begins" — exactly 2 hops from Rises (Rises → Dark Knight → Begins).

**(d)** 3 results: "The Dark Knight Rises" (0 hops = self), "The Dark Knight" (1 hop), "Batman Begins" (2 hops). The `*` operator includes zero hops (the starting node itself). The difference: `+` requires at least one traversal step; `*` includes the starting node.

</details>

---

### Q14. Inverse path — finding sequels

"What films are sequels of Batman Begins?" Instead of following `sequelOf` forward, follow it backward using the `^` operator:

In [23]:
%%sparql --file movie_kg.ttl
PREFIX ex:   <http://example.org/movie#>

SELECT ?title WHERE {
  ex:film_tt0372784 ^ex:sequelOf ?sequel .
  ?sequel ex:hasTitle ?title .
}

title          
---------------
The Dark Knight

(1 row)



**Predict:** The Dark Knight is a sequel of Batman Begins. If The Dark Knight Rises is a sequel of The Dark Knight (not directly of Begins), does it appear? (No — `^ex:sequelOf` is one hop backward.)

Run and verify. Then try `^ex:sequelOf+` for the transitive version. **Predict:** How many results now?

<details>
<summary>Solution</summary>

**One-hop inverse:** 1 result — "The Dark Knight". Dark Knight Rises does not appear because it is a sequel of Dark Knight, not directly of Begins.

**Transitive inverse (`^ex:sequelOf+`):** 2 results — "The Dark Knight" (1 hop) and "The Dark Knight Rises" (2 hops). The `+` operator follows the inverse path transitively: Begins ← Dark Knight ← Rises.

</details>

---

## Part 6 — Debug Exercise

### Q15. Bug hunt — diagnose and fix an inflated query

The following query is supposed to count the number of actors for each director's films. Run it:

In [24]:
%%sparql --file movie_kg.ttl
PREFIX ex:   <http://example.org/movie#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT ?directorName (COUNT(?actor) AS ?actorCount)
WHERE {
  ?director rdf:type ex:Director .
  ?director ex:hasName ?directorName .
  ?director ex:directed ?film .
  ?film ex:hasGenre ?genre .
  ?actor ex:actedIn ?film .
}
GROUP BY ?director ?directorName
ORDER BY DESC(?actorCount)

directorName      | actorCount
------------------+-----------
Christopher Nolan | 9         
Denis Villeneuve  | 3         
Greta Gerwig      | 2         

(3 rows)



**(a)** The counts are inflated. Explain why. (Hint: trace the bindings. The `?film ex:hasGenre ?genre` pattern multiplies each film's bindings by its genre count. Each (film, genre, actor) combination is counted, not each (film, actor) pair.)

**(b)** Fix the query. Two valid approaches: remove the `?genre` pattern (if it is not needed for the output), or use `COUNT(DISTINCT ?actor)`.

**(c)** Run the fixed version. Compare counts to the broken version. For a director with 4 films averaging 3 genres each and 2 actors per film, what did the broken query report? What does the fixed query report?

<details>
<summary>Solution</summary>

**(a)** The `?film ex:hasGenre ?genre` pattern is unnecessary for counting actors but multiplies each film's bindings by its genre count. For example, The Dark Knight has 3 genres and 1 actor (Murphy), producing 3 bindings instead of 1. The COUNT sees 3 actor appearances instead of 1.

Broken counts:
- Nolan: 9 (Inception 2 genres × 1 actor + Interstellar 2 × 1 + Oppenheimer 2 × 1 + Dark Knight 3 × 1)
- Villeneuve: 3 (Dune 3 genres × 1 actor)
- Gerwig: 2 (Barbie 1 genre × 2 actors; Lady Bird has 2 genres but no `actedIn` data → dropped)

**(b)** Fix by removing the unnecessary genre pattern:

```sparql
SELECT ?directorName (COUNT(?actor) AS ?actorCount)
WHERE {
  ?director rdf:type ex:Director .
  ?director ex:hasName ?directorName .
  ?director ex:directed ?film .
  ?actor ex:actedIn ?film .
}
GROUP BY ?director ?directorName
ORDER BY DESC(?actorCount)
```

Or keep the genre pattern but use `COUNT(DISTINCT ?actor)`.

**(c)** Fixed counts (without genre pattern):
- Nolan: 4 (Murphy in 3 films + Chalamet in 1 film = 4 actor appearances)
- Gerwig: 2 (Robbie + Gerwig in Barbie)
- Villeneuve: 1 (Chalamet in Dune)

With `COUNT(DISTINCT ?actor)`: Nolan 2, Gerwig 2, Villeneuve 1 (distinct actors, not appearances).

For the hypothetical: 4 films × 3 genres × 2 actors = 24 (broken). Fixed: 4 films × 2 actors = 8 appearances, or 2 distinct actors.

</details>

---

## Part 7 — Wikidata Enrichment

This section queries Wikidata's public SPARQL endpoint using the `%%sparql --endpoint` mode — fulfilling Chapter 8's integration promise. The exercises are bounded: one query, one link-back. Network issues should not block the rest of the lab.

### Q16. Remote query — Wikidata

Christopher Nolan's Wikidata entity is `wd:Q25191`. Query Wikidata for his birth date and birth place:

**Before running, predict:** The query uses two Wikidata property IDs (`wdt:P569` for date of birth, `wdt:P19` for place of birth) on a single entity (`wd:Q25191`). Both are 1:1 properties — a person has one birth date and one birth place. How many rows should the result have?

In [25]:
%%sparql --endpoint https://query.wikidata.org/sparql
PREFIX wd:       <http://www.wikidata.org/entity/>
PREFIX wdt:      <http://www.wikidata.org/prop/direct/>
PREFIX wikibase: <http://wikiba.se/ontology#>
PREFIX bd:       <http://www.bigdata.com/rdf#>

SELECT ?birthDate ?birthPlaceLabel WHERE {
  wd:Q25191 wdt:P569 ?birthDate .
  wd:Q25191 wdt:P19 ?birthPlace .
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en" . }
}

birthDate            | birthPlaceLabel
---------------------+----------------
1970-07-30T00:00:00Z | London         

(1 row)



Run and compare to your prediction.

Note the `SERVICE wikibase:label` line — this is Wikidata's label service, which automatically populates `rdfs:label` for entities in the query. The `?birthPlaceLabel` variable is a convention: append `Label` to any variable name and the service fills it with the entity's `rdfs:label` in the requested language. This is Q7's lesson in action — Wikidata uses the standard `rdfs:label` predicate, which is why every linked-data consumer can read its human-readable names without knowing a custom predicate.

<details>
<summary>Solution</summary>

**Predicted:** 1 row — both properties are 1:1 for a single entity.

**Actual:** 1 row — Born 1970-07-30, Place: Westminster (London). Wikidata uses `wdt:P569` for date of birth and `wdt:P19` for place of birth. The label service resolved `?birthPlace` (a Wikidata entity URI) into `?birthPlaceLabel` ("Westminster") using `rdfs:label`.

</details>

---

### Q17. Link-back — connecting Wikidata to your local graph

The Wikidata result tells you Nolan's birth date. Your local graph has `ex:person_nm0634240` but no birth date triple. Add the fact and save the updated graph:

In [26]:
from rdflib import Graph, Namespace, Literal, XSD

g = Graph()
g.parse("movie_kg.ttl")
ex = Namespace("http://example.org/movie#")

g.add((ex.person_nm0634240, ex.birthDate, Literal("1970-07-30", datatype=XSD.date)))
g.serialize("movie_kg.ttl", format="turtle")
print("Added birthDate triple to movie_kg.ttl")

Added birthDate triple to movie_kg.ttl



Now write a SPARQL query against your local graph that returns Nolan's name via `rdfs:label` (available from Q7's inference), the films he directed, and his birth date — all from one query:

**Before running, predict:** The query has four patterns. `rdfs:label` is 1:1 (one inferred label for Nolan). `ex:directed` is 1:many (6 films). `ex:hasTitle` is 1:1. `ex:birthDate` is 1:1. Which pattern determines the row count? How many rows total?

In [27]:
%%sparql --file movie_kg.ttl
PREFIX ex:   <http://example.org/movie#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?label ?filmTitle ?birthDate WHERE {
  ex:person_nm0634240 rdfs:label ?label .
  ex:person_nm0634240 ex:directed ?film .
  ?film ex:hasTitle ?filmTitle .
  ex:person_nm0634240 ex:birthDate ?birthDate .
}

(no results)



Run and compare to your prediction. This closes the loop: Wikidata contributed a fact (birth date), you added it to the local graph using the same URI (`ex:person_nm0634240`), and now it is queryable alongside the filmography data. The `rdfs:label` pattern works because of Q7's `rdfs:subPropertyOf` declaration — the same standard predicate that Wikidata uses to label `wd:Q25191` now also retrieves `ex:hasName` values from your local graph.

**Discussion:** In production, you would not manually add Wikidata facts. You would use `owl:sameAs` to link `ex:person_nm0634240` to `wd:Q25191` and query across both graphs with federated SPARQL (the `SERVICE` keyword). This is beyond the scope of this lab but is the natural next step for full linked-data integration.

<details>
<summary>Solution</summary>

**Predicted:** 6 rows — the `ex:directed` pattern is 1:many (6 films); all other patterns are 1:1, so the row count equals the number of directed edges.

**Actual: 6 rows** — one per film Nolan directed, each with the same label and birth date:

| ?label | ?filmTitle | ?birthDate |
|---|---|---|
| "Christopher Nolan" | "Inception" | "1970-07-30" |
| "Christopher Nolan" | "Interstellar" | "1970-07-30" |
| "Christopher Nolan" | "Oppenheimer" | "1970-07-30" |
| "Christopher Nolan" | "Batman Begins" | "1970-07-30" |
| "Christopher Nolan" | "The Dark Knight" | "1970-07-30" |
| "Christopher Nolan" | "The Dark Knight Rises" | "1970-07-30" |

The label comes from `rdfs:label` (inferred from `ex:hasName` via `rdfs:subPropertyOf`). The birth date comes from the triple added in the previous cell. Both are queryable alongside the original filmography data — integration via URI alignment.

</details>

---

## Submission Checklist

Before submitting, verify you have:

- [ ] Manual bindings-table traces with step-by-step pattern evaluation (Q1–Q2)
- [ ] Binding-count predictions recorded *before* running for all Part 2 queries (Q3–Q6)
- [ ] URI opacity discussion and `rdfs:label` inference demonstration (Q7)
- [ ] COUNT inflation diagnosis: broken vs fixed counts with explanation (Q8)
- [ ] Genre statistics with both COUNTs using DISTINCT (Q9)
- [ ] OPTIONAL enrichment with predictions of bound/unbound rows (Q10)
- [ ] FILTER trap demonstration with explanation and fix (Q11)
- [ ] FILTER NOT EXISTS with open-world interpretation (Q12)
- [ ] Property path traversals — one-hop, transitive, bounded, and inverse (Q13–Q14)
- [ ] Bug hunt: explanation of multiplicity bug and corrected query (Q15)
- [ ] Wikidata query result recorded (Q16)
- [ ] Link-back query combining local and Wikidata-sourced data (Q17)